# 📗 부록 02: GitHub Actions로 테스트를 자동 실행하고 워크플로 설계하기

교안 02에서는 pull request마다 평가와 회귀 테스트를 돌리는 워크플로를 읽고 실행 결과를 확인했습니다. 이 부록은 GitHub Actions를 처음부터 다룹니다.

이 노트북과 같은 폴더의 연습 프로젝트(`pytest_ci_practice/`)를 GitHub 저장소에 올려 워크플로를 직접 돌려 보고, 요구사항을 보고 워크플로를 설계해 봅니다. 연습 프로젝트는 API 키가 필요 없는 pytest 테스트만 실행하므로 비용 없이 따라 할 수 있습니다.

**학습 목표**

- GitHub Actions의 구성 요소(이벤트, 워크플로, 작업, 단계, 러너, 액션)를 구분합니다.
- 워크플로 파일을 저장소에 올리고, 실행 결과와 단계별 로그를 읽습니다.
- 실패한 pull request의 검사를 읽고, 직접 실행과 비밀 변수를 사용합니다.
- 정해진 순서(언제, 무엇을, 어떤 순서로, 어떤 컴퓨터에서, 실패했을 때, 권한과 비용)로 워크플로를 설계합니다.
- (선택) Ruff로 코드 스타일을 검사·정리하고, 그 검사를 워크플로에 넣습니다.
- (선택) pull request마다 LLM이 바뀐 코드를 리뷰해 댓글로 남기는 작업을 워크플로에 더합니다.

<img src="../images/ap_00_actions_parts.png" width="1050" alt="push, pull_request, schedule, workflow_dispatch 같은 이벤트가 .github/workflows 의 워크플로를 실행하고, 워크플로 안의 작업이 러너 위에서 단계(uses 액션, run 명령)를 차례로 실행해 체크 표시, 로그, 산출물을 남기는 구조입니다.">

## 1. GitHub Actions 구성 요소

### 1-1. GitHub Actions가 하는 일

**GitHub Actions**는 저장소에 정해 둔 이벤트가 생기면(예: 브랜치에 커밋을 올리는 `push`, pull request를 열거나 커밋을 더하는 `pull_request`) GitHub 서버의 가상 컴퓨터에서 정해 둔 명령을 자동으로 실행하는 기능입니다. 변경된 코드를 올릴 때마다 테스트를 자동으로 돌려 문제를 미리 잡는 방식을 **CI**(지속적 통합)라고 하고, GitHub Actions는 CI를 만드는 도구 가운데 하나입니다.

### 1-2. 여섯 가지 용어

| 용어 | 뜻 | 연습 프로젝트의 예 |
|---|---|---|
| 이벤트(event) | 워크플로를 실행하게 만드는 일 | `push`, `pull_request`, 직접 실행(`workflow_dispatch`) |
| 워크플로(workflow) | 언제, 무엇을 실행할지 적은 YAML 파일 하나 | `.github/workflows/tests.yml` |
| 작업(job) | 러너 한 대에서 차례로 실행할 단계 묶음 | `pytest` |
| 단계(step) | 작업 안의 일 하나. 액션 하나(`uses`) 또는 명령 하나(`run`) | `uv run pytest -v tests` |
| 러너(runner) | 작업을 실행하는 가상 컴퓨터 | `ubuntu-latest` |
| 액션(action) | 남이 만들어 공개해 둔 단계. `uses`로 가져다 씀 | `actions/checkout`(저장소 코드 내려받기) |

워크플로 하나에 작업이 여러 개일 수 있고, 작업마다 빈 러너가 새로 켜집니다. 그래서 작업마다 저장소 받기와 라이브러리 설치를 다시 해야 합니다.

### 1-3. 무료로 쓸 수 있는 범위

GitHub Actions 요금은 저장소의 공개 여부와 러너 종류로 정해집니다.

- **공개 저장소:** GitHub가 제공하는 표준 러너(`ubuntu-latest`, `windows-latest`, `macos-latest`)는 시간 제한 없이 무료입니다.
- **비공개 저장소(GitHub Free 계정):** 한 달에 2,000분까지 무료입니다. 산출물(artifact) 저장 공간은 500MB, 캐시는 저장소마다 10GB이고, 무료 분량은 결제 주기마다 새로 채워집니다. GitHub Pro 계정은 3,000분입니다.
- **러너 종류:** 분당 요금이 Linux(2코어) 0.006달러, Windows 0.010달러, macOS 0.062달러로, 같은 시간을 돌려도 Windows·macOS 러너가 더 비쌉니다.
- **무료 분량을 다 쓰면:** 결제 수단을 등록하지 않았으면 그달 남은 기간 동안 실행이 막히고 요금은 나가지 않습니다. 결제 수단을 등록했다면 넘은 만큼 청구되므로 예산(Budgets)을 설정해 둡니다.
- **실행 제한:** GitHub Free 계정은 동시에 도는 작업이 20개(macOS는 5개)까지이고, 작업 하나는 최대 6시간까지 돕니다.
- **직접 준비한 컴퓨터(self-hosted runner):** 러너 요금이 없습니다.

이 부록의 연습 프로젝트는 `timeout-minutes: 10`으로 한 번 실행이 최대 10분이라, 비공개 저장소로 올려도 무료 분량 안에서 따라 할 수 있습니다. 요금은 바뀔 수 있으니 공식 문서([GitHub Actions 요금](https://docs.github.com/en/billing/concepts/product-billing/github-actions), [사용 제한](https://docs.github.com/en/actions/reference/limits))에서 확인합니다.

### 1-4. GitLab CI와 비교

GitLab 저장소에서는 **GitLab CI**가 같은 일을 합니다.

- **공통점:** YAML 파일에 언제·무엇을 실행할지 적고, 러너가 작업을 실행합니다. 명령의 종료 코드가 0이 아니면 실패로 표시하고, API 키는 저장소 설정의 비밀 변수로 넘깁니다.
- **설정 파일:** GitHub는 `.github/workflows/` 아래에 워크플로 파일을 여러 개 둘 수 있고, GitLab은 저장소 맨 위의 `.gitlab-ci.yml` 하나에 적습니다.
- **실행 순서:** GitHub는 작업 안의 단계(step)를 차례로 실행하고 작업 사이 순서는 `needs`로 정합니다(5절). GitLab은 작업을 스테이지(stage)로 묶어 스테이지 순서대로 실행하고, 작업의 명령은 `script` 목록에 적습니다.
- **이름:** 변경 요청은 GitHub의 pull request, GitLab의 merge request이고, 실행 한 번은 GitHub의 워크플로 실행(workflow run), GitLab의 파이프라인(pipeline)입니다. 비밀 변수는 GitHub의 Secrets, GitLab의 CI/CD 변수입니다.
- **재사용:** GitHub는 남이 만든 액션을 `uses`로 가져다 쓰고, GitLab은 다른 설정 파일을 `include`로 가져옵니다.
- **무료 분량:** GitLab.com 무료 계정은 한 달에 400분(compute minutes)입니다.

## 2. 첫 워크플로 만들기

### 2-1. 워크플로 파일 읽기

GitHub는 저장소의 `.github/workflows/` 폴더에 있는 YAML 파일만 워크플로로 읽습니다. YAML은 들여쓰기로 구조를 나타내므로, 같은 깊이로 들여 쓴 줄이 한 묶음입니다.

연습 프로젝트의 `tests.yml`은 main 브랜치에 올릴 때, pull request를 열거나 커밋을 더할 때, Actions 화면에서 버튼을 누를 때 테스트를 돌립니다. 줄마다 오른쪽 주석에 뜻을 적었습니다.

`.github/workflows/tests.yml`

```yaml
name: tests                                       # Actions 탭과 검사 영역에 보일 워크플로 이름

on:                                               # 이 워크플로를 실행할 이벤트
  push:                                           # 브랜치에 커밋을 올릴 때
    branches: [main]                              # 그중 main 브랜치에 올릴 때만
  pull_request:                                   # pull request를 열거나 커밋을 더할 때
  workflow_dispatch:                              # Actions 탭의 Run workflow 버튼을 누를 때

jobs:                                             # 할 작업 목록(작업마다 새 러너에서 따로 실행)
  pytest:                                         # 작업 이름. 실행 화면과 검사 영역에 이 이름으로 보임
    runs-on: ubuntu-latest                        # 러너: GitHub가 빌려주는 최신 우분투 가상 컴퓨터
    timeout-minutes: 10                           # 10분이 지나도 안 끝나면 작업을 멈춤
    steps:                                        # 차례로 실행할 단계(앞 단계가 실패하면 뒤는 건너뜀)
      - name: 저장소 받기                         # 단계 이름. 실행 로그에 이 이름으로 보임
        uses: actions/checkout@v7                 # 남이 만든 액션 실행: 저장소 코드를 러너로 내려받음
      - name: uv 설치
        uses: astral-sh/setup-uv@v10.2.0          # 남이 만든 액션 실행: 러너에 uv 설치(@ 뒤는 버전)
        with:                                     # 액션에 넘기는 값(받는 값은 액션 README에 있음)
          python-version: "3.12"                  # uv가 쓸 파이썬 버전
      - name: 라이브러리 설치
        run: uv sync --locked                     # 명령 실행: uv.lock에 적힌 버전 그대로 설치
      - name: 비밀 변수 확인
        env:                                      # 이 단계에서만 쓰는 환경변수
          DEMO_TOKEN: ${{ secrets.DEMO_TOKEN }}   # 등록한 비밀 변수의 값(4-2에서 등록)
        run: |                                    # | 다음 줄부터 들여 쓴 여러 줄을 차례로 실행
          echo "DEMO_TOKEN 값: $DEMO_TOKEN"
          echo "DEMO_TOKEN 길이: ${#DEMO_TOKEN}"  # ${#변수}는 셸에서 값의 글자 수
      - name: 테스트
        run: uv run pytest -v tests               # 명령 실행: 테스트. 하나라도 실패하면 작업 실패
```

단계는 `uses`와 `run` 두 가지로 씁니다.

- **`uses`:** 남이 만들어 공개해 둔 **액션**을 가져다 그 일을 시킵니다. `actions/checkout@v7`은 GitHub(`actions`)가 만든 `checkout` 액션의 v7을 쓴다는 뜻이고, 이 액션이 저장소 코드를 빈 러너로 내려받습니다. 이 단계가 없으면 러너에 코드가 없어 테스트를 돌릴 수 없습니다.
- **`run`:** 러너의 셸에서 명령을 그대로 실행합니다. 내 컴퓨터 터미널에서 치던 명령(`uv sync --locked`, `uv run pytest -v tests`)을 적는다고 생각하면 됩니다.

### 2-2. 저장소에 올리기

git의 `add`·`commit`·`push`와 pull request는 앞선 단원에서 익혔습니다. GitHub에서 새 저장소를 만든 뒤, 저장소 최상위 폴더에서 연습 프로젝트 폴더로 이동해 그 저장소로 올립니다.

- README 같은 파일을 넣지 않은 빈 저장소로 만들어야 첫 push가 거절되지 않습니다.
- 비공개 저장소도 되지만, 6절의 병합 막기 규칙까지 해 보려면 공개 저장소로 만듭니다.

`.github/workflows/tests.yml`이 함께 올라가면 워크플로가 등록되고, main 브랜치에 올렸으므로 첫 실행이 바로 시작됩니다.

```bash
cd day61_평가자동화_MLflow/부록_02_GitHub_Actions_기초/pytest_ci_practice
git init -b main
git add .
git commit -m "인용 검사 연습 프로젝트와 tests 워크플로 추가"
git remote add origin https://github.com/<계정>/<저장소>.git
git push -u origin main
```

### 2-3. 실행 결과와 로그 확인

저장소 위쪽 **Actions** 탭에 실행이 한 줄씩 쌓입니다. 초록 체크는 성공, 빨간 X는 실패입니다. 각 줄에는 실행 이름(커밋 메시지나 pull request 제목), 워크플로 이름, 이벤트가 보입니다.

<img src="../images/ap_01_actions_runs.png" width="1050" alt="Actions 탭의 실행 목록. main 에 올린 실행과 직접 실행은 성공(초록 체크), 실패하는 pull request 의 실행은 빨간 X로 표시됩니다.">

- 누를 곳: 저장소 위쪽 Actions 탭
- 주소: `https://github.com/<계정>/<저장소>/actions`

그림은 이 부록을 끝까지 진행한 뒤의 목록입니다. 맨 아래 줄이 방금 올린 첫 실행이고, 빨간 사각형의 두 줄은 3절의 실패하는 pull request와 7절에서 올린 `ci` 워크플로의 실행입니다.

실행 이름을 누르면 작업 목록이 나오고, 작업 이름(`pytest`)을 누르면 단계별 로그가 보입니다. 단계 이름 왼쪽 화살표를 누르면 그 단계의 출력이 펼쳐집니다. 테스트 단계에는 부록 01에서 본 pytest 출력이 그대로 나옵니다.

<img src="../images/ap_02_job_steps.png" width="1050" alt="pytest 작업의 단계 목록(저장소 받기, uv 설치, 라이브러리 설치, 비밀 변수 확인, 테스트)과 펼친 테스트 단계의 pytest 출력, 마지막 줄의 passed 요약입니다.">

- 누를 곳: Actions > 실행 이름 > 왼쪽 작업 이름(pytest) > 테스트 단계 펼치기
- 주소: `https://github.com/<계정>/<저장소>/actions/runs/<실행 ID>/job/<작업 ID>`

## 3. 실패한 pull request 읽기

pull request 화면 아래쪽 검사 영역에서는 병합하기 전에 테스트가 통과했는지 볼 수 있습니다. 일부러 테스트가 실패하는 pull request를 하나 열어 봅니다.

### 3-1. pull request 화면의 검사 영역

새 브랜치에서 `citation_check.py`의 가운뎃점 목록 `["･", "∙", "・", "•"]`에서 **두 번째 항목** `"∙"`를 지우고 올립니다. 네 번째 항목 `"•"`와 모양이 비슷하니 위치로 찾습니다.

```bash
git switch -c refactor/simplify-normalize
# 편집기로 citation_check.py 를 열어 목록의 두 번째 항목 "∙" 를 지우고 저장합니다.
git commit -am "가운뎃점 정규화 목록 정리"
git push -u origin refactor/simplify-normalize
```

GitHub 저장소 화면에 뜨는 **Compare & pull request**를 눌러 main으로 가는 pull request를 엽니다. 이 pull request는 병합하지 않습니다.

`pull_request` 이벤트로 워크플로가 돌고, pull request 화면 아래 검사 영역에 결과가 붙습니다. `test_middle_dot_shapes_are_unified`가 `"∙"`를 쓰므로 이 테스트가 실패합니다.

<img src="../images/ap_04_pr_checks_failed.png" width="1050" alt="pull request 화면 아래쪽 검사 영역에 tests 워크플로의 pytest 검사가 실패로 표시된 모습입니다.">

- 누를 곳: 저장소 위쪽 Pull requests 탭 > 방금 연 pull request > Conversation 탭 맨 아래
- 주소: `https://github.com/<계정>/<저장소>/pull/<번호>`

검사가 여러 개면 실패한 검사와 성공한 검사의 개수가 따로 나오고, 실패한 검사가 목록 위에 펼쳐집니다.

### 3-2. 실패 로그 읽기

검사 옆 **Details**를 누르면 실패한 작업의 로그로 갑니다. GitHub는 실패한 단계를 자동으로 펼쳐 둡니다.

읽는 순서는 부록 01과 같습니다. `short test summary info`의 `FAILED` 줄로 실패한 테스트를 찾고, 위쪽의 `E` 줄에서 두 값이 어떻게 다른지 봅니다.

<img src="../images/ap_05_failed_log.png" width="1050" alt="실패한 테스트 단계 로그. test_middle_dot_shapes_are_unified 가 FAILED로 나오고, E 줄에 기대한 문자열과 실제 문자열의 차이가 보입니다.">

- 누를 곳: 검사 영역의 Details > 펼쳐진 테스트 단계
- 주소: `https://github.com/<계정>/<저장소>/actions/runs/<실행 ID>/job/<작업 ID>`

지금은 검사가 실패해도 병합 버튼이 눌립니다. 실패한 검사로 병합을 막는 규칙은 6절에서 설정합니다.

## 4. 직접 실행과 비밀 변수

### 4-1. 화면에서 직접 실행하기

`on`에 `workflow_dispatch`가 있으면 Actions 화면의 버튼으로 워크플로를 실행할 수 있습니다. 코드를 바꾸지 않고 다시 확인하고 싶을 때 씁니다.

<img src="../images/ap_06_run_workflow.png" width="1050" alt="Actions 탭에서 tests 워크플로를 고르고 Run workflow 버튼을 눌러 브랜치 선택 창이 열린 모습입니다.">

- 누를 곳: Actions > 왼쪽 워크플로 목록에서 tests > 오른쪽 Run workflow > 브랜치 고르고 실행
- 주소: `https://github.com/<계정>/<저장소>/actions/workflows/tests.yml`

### 4-2. 비밀 변수 등록과 사용

API 키처럼 남에게 보이면 안 되는 값은 YAML에 쓰지 않고 저장소의 **비밀 변수**(Secrets)로 등록합니다. 등록한 값은 등록한 사람도 다시 볼 수 없고, 워크플로에서는 `${{ secrets.이름 }}`으로만 꺼내 씁니다.

<img src="../images/ap_07_secrets_page.png" width="1050" alt="저장소 Settings 의 Secrets and variables > Actions 화면에서 Repository secrets 목록에 DEMO_TOKEN 이 등록된 모습입니다.">

- 누를 곳: 저장소 위쪽 Settings > 왼쪽 Secrets and variables > Actions > New repository secret
- 주소: `https://github.com/<계정>/<저장소>/settings/secrets/actions`

`DEMO_TOKEN`의 값에는 실제 키가 아닌 연습용 글자를 넣습니다.

첫 실행 때는 `DEMO_TOKEN`이 없어서 로그에 빈 값과 길이 0이 찍혔습니다. 등록한 뒤 4-1의 Run workflow로 다시 실행합니다.

`tests.yml`의 "비밀 변수 확인" 단계는 등록한 `DEMO_TOKEN`을 일부러 출력합니다. GitHub는 로그에 비밀 변수의 값이 나오면 `***`로 가립니다. 길이는 숫자라 그대로 보이므로, 값이 실제로 전달됐는지는 이렇게 확인할 수 있습니다.

<img src="../images/ap_03_secret_masked.png" width="1050" alt="비밀 변수 확인 단계의 로그. DEMO_TOKEN 값이 *** 로 가려지고 길이만 숫자로 보입니다.">

- 누를 곳: Actions > 실행 이름 > pytest > 비밀 변수 확인 단계 펼치기
- 주소: `https://github.com/<계정>/<저장소>/actions/runs/<실행 ID>/job/<작업 ID>`

가려 준다고 비밀 변수를 출력해도 되는 것은 아닙니다. 값을 바꿔 출력하면(앞 몇 글자만 자르는 등) 가려지지 않으므로, 실제 워크플로에서는 비밀 변수를 출력하지 않습니다. 이 단계는 확인용이니 확인한 뒤에는 지웁니다.

## 5. 워크플로 설계하기

워크플로를 설계한다는 것은 YAML을 쓰기 전에 다음 여섯 가지를 차례로 정하는 일입니다. 정한 내용은 그대로 YAML의 키가 됩니다.

<img src="../images/ap_00_workflow_design.png" width="1050" alt="워크플로 설계 순서 여섯 단계. 1 언제 돌릴지(on), 2 무엇을 할지(jobs, steps), 3 어떤 순서로(needs), 4 어떤 컴퓨터에서(runs-on, matrix), 5 실패했을 때(if, 산출물), 6 권한과 비용(permissions, timeout).">

아래는 이 절에서 예로 드는 워크플로의 뼈대입니다. 빠른 단위 테스트(`unit`)를 먼저 돌리고, 통과하면 느린 LLM 평가(`eval`)를 돌립니다. 오른쪽 주석의 번호가 설계 순서입니다.

```yaml
name: ci
on: ...                                   # 1. 언제 돌릴지
permissions: ...                          # 6. 권한
concurrency: ...                          # 6. 같은 브랜치의 이전 실행 취소
jobs:                                     # 2. 무엇을 할지: 작업 목록
  unit:                                   # 작업 하나마다 새 러너 하나
    strategy: ...                         # 4. 여러 환경에서(matrix)
    runs-on: ubuntu-latest                # 4. 어떤 컴퓨터에서
    steps:                                # 2. 이 작업 안에서 차례로 할 단계
      - uses: ...                         # 남이 만든 액션 실행
      - run: ...                          # 명령 실행
  eval:
    needs: unit                           # 3. 어떤 순서로: unit 이 성공한 뒤 시작
    timeout-minutes: 15                   # 6. 시간 제한
    steps:
      - run: ...
      - if: always()                      # 5. 실패했을 때도 실행
        uses: actions/upload-artifact@v7  # 5. 결과 파일을 산출물로 저장
```

### 5-1. 언제 돌릴지: `on`

| 이벤트 | 실행되는 때 | 알맞은 일 |
|---|---|---|
| `push` | 브랜치에 커밋을 올릴 때 | main에 들어간 코드가 여전히 통과하는지 확인 |
| `pull_request` | pull request를 열거나 커밋을 더할 때 | 병합 전에 변경을 검사(회귀 게이트) |
| `schedule` | 정해 둔 시각(`cron`, UTC 기준) | 비싸서 매번 못 돌리는 전체 평가 |
| `workflow_dispatch` | Actions 화면에서 버튼을 누를 때 | 코드 변경 없이 다시 확인 |

```yaml
on:
  push:
    branches: [main]           # main 브랜치에 올릴 때만
    paths-ignore: ["docs/**"]  # docs 폴더만 바뀐 커밋에는 돌지 않음
  pull_request:                # 모든 pull request(열 때와 커밋을 더할 때)
  schedule:
    - cron: "0 0 * * 1"        # 매주 월요일 0시(UTC), 한국 시각 오전 9시
  workflow_dispatch:           # Actions 탭의 버튼으로 직접 실행
```

- `branches`는 브랜치로, `paths`·`paths-ignore`는 바뀐 파일로 실행을 거릅니다.
- `push`와 `pull_request`를 둘 다 걸 때 `push`에 `branches: [main]`이 없으면, pull request 브랜치에 커밋을 올릴 때 같은 테스트가 두 번 돕니다.

### 5-2. 무엇을 할지: 작업(`jobs`)과 단계(`steps`)

먼저 할 일을 순서대로 적고, 그것을 작업과 단계로 나눕니다.

1. 할 일 적기: 저장소 받기 → uv 설치 → 라이브러리 설치 → 단위 테스트 → LLM 평가
2. 작업 나누기: 따로 성공·실패를 보고 싶은 일, 다른 컴퓨터에서 돌릴 일, 비싸서 뒤로 미룰 일은 작업을 나눕니다. 여기서는 단위 테스트(`unit`)와 평가(`eval`) 두 작업입니다.
3. 단계 채우기: 작업마다 빈 러너가 새로 켜지므로, 작업마다 저장소 받기와 설치 단계를 다시 적습니다.

```yaml
jobs:
  unit:                                   # 작업 1: 빠른 단위 테스트
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v7         # 액션: 저장소 코드를 러너로 내려받기
      - uses: astral-sh/setup-uv@v10.2.0  # 액션: 러너에 uv 설치
      - run: uv sync --locked             # 명령: 라이브러리 설치
      - run: uv run pytest -v tests       # 명령: 단위 테스트
  eval:                                   # 작업 2: 느린 LLM 평가. 새 러너라 받기와 설치를 다시 함
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v7
      - uses: astral-sh/setup-uv@v10.2.0
      - run: uv sync --locked
      - run: uv run python run_eval.py    # 명령: 교안 02처럼 골든셋 평가
```

- **단계(step):** 한 작업 안의 단계는 같은 러너에서 위에서부터 차례로 실행됩니다. 앞 단계가 실패하면 뒤 단계는 건너뜁니다.
- **`uses`(액션):** 남이 만들어 공개해 둔 액션을 `만든 곳/액션 이름@버전`으로 가져다 씁니다. 자주 쓰는 것은 `actions/checkout`(저장소 코드 내려받기), `astral-sh/setup-uv`(uv 설치), `actions/upload-artifact`(파일을 산출물로 올리기)입니다. 액션이 받는 값(`with`)은 그 액션 저장소의 README에 있습니다.
- **버전:** `@v7`은 v7 가운데 가장 최신을, `@v10.2.0`은 그 버전 하나를 씁니다. 공식 문서는 커밋 SHA로 고정하는 방법이 가장 안전하다고 안내합니다.
- **`run`(명령):** 러너의 셸에서 명령을 그대로 실행합니다.

### 5-3. 어떤 순서로: `needs`

작업끼리는 기본으로 **동시에** 돕니다. 순서가 필요하면 뒤 작업에 `needs`를 씁니다.

```yaml
jobs:
  unit:          # 먼저 도는 작업
    ...
  eval:
    needs: unit  # unit 이 성공해야 시작. unit 이 실패하면 eval 은 건너뜀
    ...
```

- `needs`에 적은 작업이 성공해야 시작하고, 그 작업이 실패하거나 건너뛰어지면 이 작업도 건너뜁니다.
- 앞 작업이 matrix로 여러 환경에서 돌면, 모든 환경이 성공해야 시작합니다.
- 빠르고 싼 검사를 앞에, 느리고 비싼 검사를 뒤에 두면, 앞에서 실패한 변경에 비싼 평가를 돌리지 않습니다.

### 5-4. 어떤 컴퓨터에서: `runs-on`과 `matrix`

`runs-on`은 러너 종류(`ubuntu-latest`, `windows-latest`, `macos-latest`)를 정합니다. 같은 작업을 여러 환경에서 돌리려면 `strategy.matrix`에 값 목록을 주고 `${{ matrix.이름 }}`으로 꺼내 씁니다.

```yaml
  unit:
    strategy:
      fail-fast: false                                  # 한 환경이 실패해도 나머지 환경은 끝까지 실행
      matrix:
        python-version: ["3.11", "3.12"]                # 값마다 같은 작업을 따로 실행(작업 2개가 생김)
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v7
      - uses: astral-sh/setup-uv@v10.2.0
        with:
          python-version: ${{ matrix.python-version }}  # 지금 도는 matrix 값을 꺼내 씀
      - run: uv run pytest -v tests
```

- 운영체제를 바꿔 돌리려면 matrix에 운영체제 목록을 주고 `runs-on`에 그 값을 넣습니다(7절 `ci.yml`에서 씁니다). 같은 테스트라도 파일 인코딩·경로·줄바꿈 처리가 운영체제마다 달라 한쪽에서만 실패할 수 있으므로, 배포할 PC의 운영체제에서도 따로 돌려 확인합니다.
- `fail-fast`의 기본값은 `true`라서, 한 환경이 실패하면 아직 도는 나머지 환경을 취소합니다. 어디서 실패했는지 모두 보려면 `false`로 둡니다.

### 5-5. 실패했을 때: `if`와 산출물

단계는 기본으로 **앞 단계가 모두 성공했을 때만** 실행됩니다. 그런데 테스트가 실패했을 때야말로 결과 파일이 필요합니다. 그래서 먼저 조건(`if`)으로 저장 단계가 실패 뒤에도 돌게 하고, 그 단계에서 결과(산출물)를 올립니다.

```yaml
      - name: 평가
        run: uv run python run_eval.py    # 평가 결과를 reports/ 에 남긴다고 가정. 실패하면 다음 단계는 기본으로 건너뜀
      - name: 평가 결과 저장
        if: always()                      # 조건: 앞 단계가 실패해도 이 단계는 실행
        uses: actions/upload-artifact@v7  # 결과: 파일을 실행 요약 화면의 Artifacts 에 올리는 액션
        with:
          name: eval-reports              # 산출물 이름
          path: reports/                  # 올릴 폴더
          retention-days: 14              # 보관 일수
```

- `if: always()`는 앞 단계가 실패해도 실행합니다. 실패했을 때만 실행하려면 `if: failure()`를 씁니다.
- 올린 산출물은 실행 요약 화면 아래 **Artifacts**에서 내려받습니다. 보관 일수가 지나면 지워집니다.

### 5-6. 권한과 비용: `permissions`, `timeout-minutes`, `concurrency`, 캐시

```yaml
permissions:
  contents: read                                         # 저장소 코드는 읽기만 허용(워크플로가 저장소를 고칠 일이 없을 때)

concurrency:
  group: eval-${{ github.ref }}                          # 같은 브랜치의 실행끼리 한 묶음(github.ref 는 실행을 일으킨 브랜치)
  cancel-in-progress: true                               # 새 실행이 오면 진행 중인 이전 실행을 취소

jobs:
  eval:
    timeout-minutes: 15                                  # 응답 없이 멈춘 작업도 15분이면 끝냄
    steps:
      - uses: astral-sh/setup-uv@v10.2.0
        with:
          enable-cache: true                             # 내려받은 라이브러리를 저장해 다음 실행에서 다시 씀
      - run: uv run python run_eval.py
        env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}  # 비밀 변수는 그것이 필요한 단계에만 넘김
```

- 비밀 변수는 다른 사람이 저장소를 복사해 간 사본(fork)에서 온 pull request에는 전달되지 않습니다.
- 비용은 1-3의 무료 한도에서 빠집니다. matrix 값이 늘어날수록, Windows·macOS 러너를 쓸수록 한도가 빨리 줄어듭니다.

## 6. 실패한 검사로 병합 막기

지금은 검사가 실패해도 pull request를 병합할 수 있습니다. 저장소 설정의 규칙(Rulesets)에 **필수 검사**를 넣으면, 그 검사가 통과해야만 main에 병합됩니다.

필수 검사의 이름은 워크플로의 **작업 이름**입니다. 2절 `tests.yml`의 작업은 `pytest` 하나이므로 `pytest`를 필수 검사로 넣습니다.

### 6-1. 규칙 만들기

- 누를 곳: 저장소 위쪽 Settings > 왼쪽 Rules > Rulesets > New ruleset > New branch ruleset
- 입력할 것: Ruleset name에 이름(예: `main 보호`), Enforcement status는 Active, Target branches는 Add target > Include default branch
- 주소: `https://github.com/<계정>/<저장소>/settings/rules`

### 6-2. 필수 검사 넣기

아래쪽 **Require status checks to pass**를 켜고 **Add checks**를 누릅니다. 검색 창에 `pytest`를 입력하면 최근 실행에서 보고된 검사가 `GitHub Actions` 표시와 함께 나옵니다. 이것을 고릅니다.

<img src="../images/ap_09_pytest_add.png" width="1050" alt="Add checks 창의 검색 칸에 pytest 를 입력하자 GitHub Actions 표시가 붙은 pytest 검사가 나온 모습입니다.">

목록에 검사가 보이지 않으면 그 워크플로가 아직 한 번도 돌지 않은 것입니다. 한 번 실행한 뒤 다시 검색합니다.

**Required status checks**에 `pytest`가 보이면 맨 아래 **Create**를 눌러 저장합니다. 이제 3절처럼 `pytest`가 실패한 pull request는 병합 버튼이 막힙니다.

<img src="../images/ap_09_pytest.png" width="1050" alt="Require status checks to pass 를 켜고 Required status checks 에 pytest 검사를 넣은 모습입니다.">

규칙은 공개 저장소이거나 유료 요금제를 쓰는 저장소에서 적용됩니다. 무료 계정의 비공개 저장소에서는 규칙을 저장해도 아래처럼 적용되지 않는다는 경고가 나오고 병합은 막히지 않습니다.

<img src="../images/ap_09_ruleset_warning.png" width="1050" alt="Rulesets 화면 위쪽에 비공개 저장소에서는 GitHub Team 계정으로 옮기기 전까지 규칙이 적용되지 않는다는 경고가 나온 모습입니다.">

필수 검사로 넣은 워크플로에 `paths`나 `paths-ignore`를 걸면, 바뀐 파일이 조건에 맞지 않아 검사가 돌지 않은 pull request는 "대기 중"으로 남아 병합할 수 없습니다.

## 7. 요구사항으로 워크플로 설계하기

`tests.yml`은 작업 하나로 모든 테스트를 돌립니다. 연습 프로젝트가 커지면서 팀이 다음 요구사항을 정했습니다.

1. main 브랜치에 올릴 때, 모든 pull request, 화면에서 직접 실행할 때 돈다.
2. 같은 브랜치에 새 실행이 오면 진행 중인 이전 실행은 취소한다.
3. 단위 테스트(`tests/test_normalize.py`, `tests/test_check_citation.py`)는 Ubuntu와 Windows 두 운영체제에서 모두 통과해야 한다. 한 운영체제에서 실패해도 다른 운영체제의 실행은 끝까지 돈다.
4. 게이트 규칙 테스트(`tests/test_gate_rules.py`)는 단위 테스트가 두 운영체제에서 모두 통과한 뒤에만 돈다.
5. 게이트 규칙 테스트가 실패해도 결과 파일(`reports/`)을 산출물 `gate-reports`로 7일 보관한다.

**실습 진행 방법**

- 7-1에서 설계 파일을 요구사항과 대조하고, 7-2에서 저장소에 올려 실행하고, 7-3에서 6절의 필수 검사를 새 워크플로에 맞게 바꿉니다.
- `tests/test_gate_rules.py`에는 교안 02의 게이트 규칙 함수(`gate_rules.py`)를 확인하는 테스트가 미리 들어 있습니다. 두 작업은 같은 `tests/` 폴더에서 실행할 테스트 파일을 골라 나눕니다.

### 7-1. 설계 파일 읽기

`pytest_ci_practice/design/ci.yml`은 요구사항을 반영한 완성 워크플로입니다. 요구사항마다 `# 요구사항 N` 주석을 달았으니 5절의 어느 설정을 썼는지 대조하며 읽습니다. 8절의 Ruff 검사 단계(`unit-tests` 작업 끝)와 9절의 `ai-review` 작업도 들어 있습니다(`[Ruff 추가]`, `[AI 리뷰 추가]` 주석).

```yaml
# 부록 02 7절 요구사항 1~5를 반영한 워크플로입니다.
# 8절의 Ruff 검사 단계와 9절의 AI 리뷰 작업([Ruff 추가], [AI 리뷰 추가] 주석)까지 들어 있습니다.
# .github/workflows/ci.yml 로 복사해 저장소에 올립니다.
name: ci

on:                                               # 요구사항 1: main 에 올릴 때, 모든 pull request, 직접 실행할 때
  push:
    branches: [main]
  pull_request:
  workflow_dispatch:

permissions:
  contents: read

concurrency:                                      # 요구사항 2: 같은 브랜치에 새 실행이 오면 이전 실행 취소
  group: ci-${{ github.ref }}
  cancel-in-progress: true

jobs:
  unit-tests:
    strategy:                                     # 요구사항 3: Ubuntu 와 Windows 에서 같은 단계 실행
      fail-fast: false                            # 한쪽이 실패해도 다른 쪽은 끝까지 실행
      matrix:
        os: [ubuntu-latest, windows-latest]
    runs-on: ${{ matrix.os }}
    timeout-minutes: 10
    steps:
      - name: 저장소 받기
        uses: actions/checkout@v7
      - name: uv 설치
        uses: astral-sh/setup-uv@v10.2.0
        with:
          python-version: "3.12"
          enable-cache: true
      - name: 라이브러리 설치
        run: uv sync --locked
      - name: 단위 테스트
        run: uv run pytest -v tests/test_normalize.py tests/test_check_citation.py
      # [Ruff 추가] 코드 스타일 검사 단계. 라이브러리 설치 단계가 dev 묶음의 ruff 까지 설치해 두었음
      - name: 문법·스타일·import 순서 검사
        if: always()                              # 앞의 테스트가 실패해도 이 검사는 실행
        run: uv run ruff check .                  # pyproject.toml 의 select 규칙을 어긴 곳이 있으면 실패
      - name: 포맷 검사
        if: always()
        run: uv run ruff format --check .         # 다시 포맷해야 할 파일이 있으면 실패(파일은 바꾸지 않음)

  gate-rules:
    needs: unit-tests                             # 요구사항 4: unit-tests 가 두 운영체제에서 모두 통과한 뒤에만 시작
    runs-on: ubuntu-latest
    timeout-minutes: 10
    steps:
      - name: 저장소 받기
        uses: actions/checkout@v7
      - name: uv 설치
        uses: astral-sh/setup-uv@v10.2.0
        with:
          python-version: "3.12"
          enable-cache: true
      - name: 라이브러리 설치
        run: uv sync --locked
      - name: 게이트 규칙 테스트
        run: uv run pytest -v tests/test_gate_rules.py --junitxml=reports/junit.xml
      - name: 테스트 결과 저장                    # 요구사항 5: 테스트가 실패해도 reports/ 를 7일 보관
        if: always()
        uses: actions/upload-artifact@v7
        with:
          name: gate-reports
          path: reports/
          retention-days: 7

  # [AI 리뷰 추가] pull request 에서 바뀐 코드를 LLM 이 리뷰해 댓글로 남기는 작업. 비밀 변수 OPENAI_API_KEY 필요
  ai-review:
    if: github.event_name == 'pull_request'      # push·직접 실행에서는 건너뜀(리뷰할 pull request 가 없음)
    runs-on: ubuntu-latest
    timeout-minutes: 10
    permissions:
      contents: read
      pull-requests: write                        # 리뷰를 pull request 댓글로 올리려면 필요
    steps:
      - name: 저장소 받기
        uses: actions/checkout@v7
        with:
          fetch-depth: 0                          # 모든 커밋을 받아야 origin/main 과 비교 가능(기본 1은 마지막 커밋만)
      - name: uv 설치
        uses: astral-sh/setup-uv@v10.2.0
        with:
          python-version: "3.12"
      - name: 라이브러리 설치
        run: uv sync --locked --group review      # 기본 묶음에 review 묶음(langchain)까지 설치
      - name: AI 리뷰 생성
        env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}   # 4-2 처럼 등록한 비밀 변수
        run: uv run python tools/ai_reviewer.py   # review.md 를 만듦
      - name: pull request 댓글로 올리기
        env:
          GH_TOKEN: ${{ github.token }}           # 실행마다 GitHub 가 주는 토큰. gh 명령이 이 값으로 로그인
        run: gh pr comment ${{ github.event.pull_request.number }} --body-file review.md
```

아래 셀로 설정을 출력해 요구사항과 하나씩 대조합니다.

In [ ]:
from pathlib import Path

import yaml

# 설계 파일을 읽습니다. yaml.safe_load 는 키 on 을 True 로 바꿔 읽으므로, 적힌 글자 그대로 읽는 BaseLoader 를 씁니다.
design_path = Path("pytest_ci_practice") / "design" / "ci.yml"
design = yaml.load(design_path.read_text(encoding="utf-8"), Loader=yaml.BaseLoader)

# 요구사항 1~5와 대조할 부분만 출력합니다.
print("트리거 설정:", design["on"])
print("동시 실행 설정:", design["concurrency"])
print("unit-tests 의 strategy:", design["jobs"]["unit-tests"]["strategy"])
print("gate-rules 의 needs:", design["jobs"]["gate-rules"].get("needs", "없음"))
print("gate-rules 의 마지막 단계:", design["jobs"]["gate-rules"]["steps"][-1])

### 7-2. 저장소에 올려 실행하기

설계가 요구사항과 맞으면 저장소에 올려 실제로 돌려 봅니다. 3절의 실패하는 pull request 브랜치에 있다면 먼저 main으로 돌아갑니다.

`design/ci.yml`을 `.github/workflows/ci.yml`로 복사하고, 같은 테스트가 두 워크플로에서 중복으로 돌지 않게 `tests.yml`은 지운 뒤 main에 올립니다.

```bash
git switch main
cp design/ci.yml .github/workflows/ci.yml
git rm .github/workflows/tests.yml
git add .
git commit -m "워크플로를 단위 테스트와 게이트 규칙 두 작업으로 나눔"
git push
```

실행 요약 화면의 작업 그림에서 `unit-tests` 두 개가 끝난 뒤 `gate-rules`가 시작되는지, 아래 **Artifacts**에 `gate-reports`가 생겼는지 확인합니다. `ai-review` 작업은 pull request일 때만 돌므로 main에 올린 이 실행에서는 건너뜁니다.

<img src="../images/ap_08_ci_graph.png" width="1050" alt="설계한 ci 워크플로의 실행 요약. unit-tests (ubuntu-latest)와 unit-tests (windows-latest) 두 작업이 끝난 뒤 gate-rules 작업으로 화살표가 이어지고, 아래 Artifacts 에 gate-reports 가 있습니다.">

- 누를 곳: Actions > 실행 이름(ci 워크플로) > Summary
- 주소: `https://github.com/<계정>/<저장소>/actions/runs/<실행 ID>`

### 7-3. 필수 검사를 세 작업으로 바꾸기

`tests.yml`을 지웠으므로 `pytest` 검사는 더 이상 돌지 않습니다. 6절 규칙에 `pytest`를 남겨 두면 모든 pull request가 "대기 중"으로 막히므로, 필수 검사를 `ci` 워크플로의 작업 세 개로 바꿉니다. matrix로 나뉜 작업은 검사 이름 괄호 안에 그 값이 붙습니다.

- `unit-tests (ubuntu-latest)`: Ubuntu에서 돈 단위 테스트
- `unit-tests (windows-latest)`: Windows에서 돈 단위 테스트
- `gate-rules`: 게이트 규칙 테스트

6절에서 만든 규칙을 열어 `pytest` 오른쪽 휴지통 아이콘으로 지운 뒤, **Add checks** 검색 창에 `unit-tests`를 입력합니다. **Suggestions** 아래 `GitHub Actions` 표시가 붙은 두 검사를 모두 고릅니다.

- 누를 곳: Settings > 왼쪽 Rules > Rulesets > 6절에서 만든 규칙 이름

<img src="../images/ap_09_ruleset_add.png" width="1050" alt="Add checks 창의 검색 칸에 unit-tests 를 입력하자 Suggestions 아래에 unit-tests (ubuntu-latest)와 unit-tests (windows-latest) 두 검사가 GitHub Actions 표시와 함께 나온 모습입니다.">

- 위쪽 `Add unit-tests (Any source)`는 고르지 않습니다. 이름만 같으면 어디서 보고한 검사든 받아들이는 항목이라, matrix 값이 붙은 실제 검사 이름과 맞지 않습니다.
- 목록에 검사가 보이지 않으면 7-2의 실행이 아직 끝나지 않은 것입니다. 끝난 뒤 다시 검색합니다.

같은 방법으로 `gate-rules`를 검색해 넣으면 **Required status checks**에 세 검사가 보입니다. 맨 아래 **Save changes**를 눌러 저장합니다. `ai-review`는 참고 의견이라 필수 검사에 넣지 않습니다.

<img src="../images/ap_09_ruleset.png" width="1050" alt="Require status checks to pass 를 켜고 Required status checks 에 unit-tests (ubuntu-latest), unit-tests (windows-latest), gate-rules 세 검사를 넣은 모습입니다.">

**왜 세 개를 모두 넣을까요?** `gate-rules`만 넣으면 단위 테스트가 실패해도 병합이 막히지 않습니다. 단위 테스트가 실패하면 `needs` 때문에 `gate-rules`가 건너뛰어지는데, GitHub는 건너뛴 작업을 성공으로 처리하기 때문입니다.

## 8. (선택) Ruff로 코드 스타일 검사 추가하기

여러 사람이 코드를 고치면 빈 줄, 공백, import 순서, 안 쓰는 import 같은 차이가 쌓입니다. **Ruff**는 이런 것을 검사하고 고치는 파이썬 도구입니다. 예전에는 세 도구를 따로 쓰던 일을 Ruff 하나가 합니다.

| 예전 도구 | 하는 일 | Ruff에서 |
|---|---|---|
| Black | 코드 포맷 정리(빈 줄, 공백, 줄 나누기) | `ruff format` |
| isort | import 순서 정리 | `ruff check`의 `I` 규칙 |
| flake8 | 코드 스타일·문법 규칙 검사(안 쓰는 import, 정의되지 않은 이름 등) | `ruff check`의 `E`, `F` 규칙 |

### 8-1. 설정 추가

연습 프로젝트의 `pyproject.toml`에 두 곳이 추가되어 있습니다(`[Ruff 추가]` 주석이 붙은 곳). `dev` 묶음은 개발할 때만 쓰는 도구 목록이고, `uv sync`가 기본으로 함께 설치합니다.

```toml
[dependency-groups]
dev = [
    "ruff==0.17.1",
]

[tool.ruff]
line-length = 120          # 한 줄 최대 길이

[tool.ruff.lint]
select = ["E", "F", "I"]   # E: 코드 스타일, F: 문법 오류·안 쓰는 import, I: import 순서
```

### 8-2. 세 역할을 실제로 보기

**`ruff check`: flake8과 isort 역할**

아래처럼 import 순서가 뒤섞이고 쓰지 않는 `json`을 불러온 파일을 검사하면 두 가지가 걸립니다.

```python
from gate_rules import metric_drop
import json


def drop_text(baseline_value, current_value):
    drop = metric_drop(baseline_value, current_value)
    return f"하락폭 {drop}"
```

```text
I001 [*] Import block is un-sorted or un-formatted
F401 [*] `json` imported but unused
Found 2 errors.
[*] 2 fixable with `--fix`.
```

- `I001`(isort 역할): 표준 라이브러리(`json`)를 먼저, 프로젝트 모듈(`gate_rules`)을 나중에 두는 순서가 아닙니다.
- `F401`(flake8 역할): `json`을 불러오고 쓰지 않았습니다.
- `[*]`는 자동으로 고칠 수 있다는 표시입니다. `uv run ruff check --fix .`를 실행하면 `json` 줄을 지우고 순서를 맞춥니다.

연습 프로젝트의 코드는 이 검사를 통과합니다(`All checks passed!`).

**`ruff format`: Black 역할**

아래처럼 쉼표와 `=` 주변에 공백이 없고, import 뒤에 빈 줄 없이 함수를 정의한 파일(`drop_text.py`)을 포맷 검사합니다. `--diff`를 붙이면 바뀔 곳을 보여 줍니다.

```python
from gate_rules import metric_drop
def drop_text(baseline_value,current_value):
    drop=metric_drop(baseline_value,current_value)
    return f'하락폭 {drop}'
```

```text
uv run ruff format --diff drop_text.py

@@ -1,4 +1,6 @@
 from gate_rules import metric_drop
-def drop_text(baseline_value,current_value):
-    drop=metric_drop(baseline_value,current_value)
-    return f'하락폭 {drop}'
+
+
+def drop_text(baseline_value, current_value):
+    drop = metric_drop(baseline_value, current_value)
+    return f"하락폭 {drop}"

1 file would be reformatted
```

- import 뒤 함수 정의 앞에 빈 줄 두 개, 쉼표 뒤와 `=` 양옆에 공백 하나를 넣고, 문자열 따옴표를 큰따옴표로 통일합니다.
- `--check`와 `--diff`는 검사만 하고 파일은 바꾸지 않습니다. 실제로 고치려면 `uv run ruff format .`처럼 옵션 없이 실행합니다.

연습 프로젝트의 코드는 이미 포맷되어 있어 이 검사도 통과합니다.

### 8-3. 워크플로에 넣기

`.github/workflows/tests.yml`의 `pytest` 작업에 Ruff 단계 두 개를 더합니다. 단계 목록 맨 끝(`테스트` 단계 다음)에 같은 들여쓰기(앞 여섯 칸)로 붙여 넣고 올립니다.

- 새 작업을 만들지 않고 `pytest` 작업에 단계로 넣었습니다. 작업마다 새 러너에서 저장소 받기와 설치를 처음부터 다시 하므로, 권한과 실행 조건이 같은 일은 한 작업에 모으면 준비를 한 번만 합니다. `라이브러리 설치` 단계(`uv sync --locked`)가 `dev` 묶음의 Ruff까지 이미 설치합니다.
- 한 작업 안에서는 앞 단계가 실패하면 뒤 단계를 건너뜁니다. `if: always()`를 붙여 테스트가 실패해도 Ruff 검사는 실행되게 했습니다.
- 규칙을 어기거나 포맷하지 않은 파일이 있으면 그 단계가 실패하고 `pytest` 검사가 실패로 표시됩니다. 6절에서 `pytest`를 필수 검사로 넣었다면 병합도 막힙니다. 로컬에서 `uv run ruff check --fix .`와 `uv run ruff format .`을 실행하고 다시 올리면 통과합니다.
- 7절 `ci.yml`에는 이 단계가 `unit-tests` 작업 끝에 이미 들어 있습니다.

`.github/workflows/tests.yml`의 `pytest` 작업 단계 끝에 추가할 단계

```yaml
      # [Ruff 추가] 코드 스타일 검사 단계. 라이브러리 설치 단계가 dev 묶음의 ruff 까지 설치해 두었음
      - name: 문법·스타일·import 순서 검사
        if: always()                              # 앞의 테스트가 실패해도 이 검사는 실행
        run: uv run ruff check .                  # pyproject.toml 의 select 규칙을 어긴 곳이 있으면 실패
      - name: 포맷 검사
        if: always()
        run: uv run ruff format --check .         # 다시 포맷해야 할 파일이 있으면 실패(파일은 바꾸지 않음)
```

## 9. (선택) AI 코드 리뷰 작업 추가하기

pull request를 열면 바뀐 코드를 LLM이 읽고 리뷰를 댓글로 남기게 합니다. 사람이 리뷰하기 전에 놓치기 쉬운 버그나 스타일 문제를 먼저 짚어 줍니다. 통과·실패를 가리는 판정이 아니라 참고 의견이므로 필수 검사에는 넣지 않습니다.

| 단계 | 하는 일 | 파일 |
|---|---|---|
| 바뀐 코드 가져오기 | `git diff origin/main...HEAD`로 이 브랜치에서 바뀐 부분만 가져옴 | `tools/ai_reviewer.py` |
| 리뷰 받기 | 프롬프트와 `ChatOpenAI`로 리뷰를 받아 `review.md`에 저장 | `tools/ai_reviewer.py` |
| 댓글로 올리기 | `gh pr comment`로 `review.md`를 pull request 댓글로 올림 | `tests.yml`의 `ai-review` 작업 |

### 9-1. 리뷰 스크립트

연습 프로젝트의 `tools/ai_reviewer.py`입니다. 워크플로가 실행하는 보조 스크립트라 `tools/` 폴더에 따로 둡니다.

- `origin/main...HEAD`(점 세 개)는 main과 갈라진 지점부터의 변경만 보여 줍니다. 그사이 main에 새로 생긴 커밋은 섞이지 않습니다.
- 바뀐 코드가 없으면 모델을 부르지 않고, 너무 긴 변경은 앞부분 10000자만 보내 비용을 줄입니다.
- LangChain은 `pyproject.toml`의 `review` 묶음에만 들어 있어 `uv sync --group review`일 때만 설치됩니다. `pytest`·`ruff` 작업의 설치는 그대로입니다.

`tools/ai_reviewer.py`

```python
"""pull request 에서 바뀐 코드(git diff)를 LLM 에게 보여 주고 받은 리뷰를 review.md 로 저장합니다.

GitHub Actions 의 tests 워크플로(ai-review 작업)가 실행하고, review.md 를 pull request 댓글로 올립니다.
로컬 실행(pytest_ci_practice 폴더에서, OPENAI_API_KEY 필요):
    git fetch origin main
    uv run --group review python tools/ai_reviewer.py
"""

import subprocess
from pathlib import Path

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# main 과 갈라진 지점부터 지금까지 이 브랜치에서 바뀐 부분만 가져옵니다(main 에 새로 생긴 커밋은 빠집니다).
diff = subprocess.run(["git", "diff", "origin/main...HEAD"], capture_output=True, text=True, check=True).stdout

if diff.strip() == "":
    review = "main 과 비교해 바뀐 코드가 없어 리뷰하지 않았습니다."
else:
    # 너무 긴 변경은 앞부분 10000자만 보내 모델 입력 길이와 비용을 줄입니다.
    diff = diff[:10000]

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "당신은 시니어 파이썬 개발자입니다. 코드 변경(diff)을 읽고 잠재적인 버그와 스타일 문제를 짚고, "
                "잘한 점도 적어 주세요. 한국어 마크다운으로 간결하게 씁니다.",
            ),
            ("human", "다음 코드 변경을 리뷰해 주세요.\n\n```diff\n{diff}\n```"),
        ]
    )
    llm = ChatOpenAI(model="gpt-4.1")
    review = (prompt | llm).invoke({"diff": diff}).content

# 워크플로의 다음 단계가 이 파일을 pull request 댓글로 올립니다.
Path("review.md").write_text(review, encoding="utf-8")
print(review)
```

### 9-2. 워크플로에 넣기

1. 4-2와 같은 방법으로 저장소 비밀 변수 `OPENAI_API_KEY`를 등록합니다.
2. `.github/workflows/tests.yml`의 맨 아래(`jobs:` 아래, `pytest` 작업 다음)에 `ai-review` 작업을 붙여 넣고 올립니다. 앞 두 칸 들여쓰기로 `pytest:`와 줄을 맞춥니다. 7절 `ci.yml`에는 이 작업이 이미 들어 있으므로 1번만 하면 됩니다.
3. pull request를 열면 `ai-review` 검사가 돌고, 끝나면 pull request에 리뷰 댓글이 달립니다.

작업에서 눈여겨볼 곳입니다.

- `if: github.event_name == 'pull_request'`: 같은 워크플로가 main에 올릴 때와 직접 실행할 때도 돌지만, 리뷰할 pull request가 없으므로 이 작업은 건너뜁니다.
- 별도 작업으로 둔 이유: 이 작업만 pull request 쓰기 권한(`permissions`), 비밀 변수, pull request일 때만 도는 조건이 필요합니다. `pytest` 작업과 나누어 권한을 이 작업에만 줍니다.
- `fetch-depth: 0`: 체크아웃은 기본으로 마지막 커밋 하나만 받습니다. 모든 커밋을 받아야 `origin/main`과 비교할 수 있습니다.
- `GH_TOKEN`: 실행마다 GitHub가 주는 토큰입니다. 러너에 미리 설치된 `gh` 명령이 이 값으로 로그인합니다.
- pull request마다 API를 한 번 부르므로 비용이 듭니다. 다른 사람의 포크에서 온 pull request에는 비밀 변수가 전달되지 않아 "AI 리뷰 생성" 단계가 실패합니다.

`.github/workflows/tests.yml`에 추가할 작업

```yaml
  # [AI 리뷰 추가] pull request 에서 바뀐 코드를 LLM 이 리뷰해 댓글로 남기는 작업. 비밀 변수 OPENAI_API_KEY 필요
  ai-review:
    if: github.event_name == 'pull_request'      # push·직접 실행에서는 건너뜀(리뷰할 pull request 가 없음)
    runs-on: ubuntu-latest
    timeout-minutes: 10
    permissions:
      contents: read
      pull-requests: write                        # 리뷰를 pull request 댓글로 올리려면 필요
    steps:
      - name: 저장소 받기
        uses: actions/checkout@v7
        with:
          fetch-depth: 0                          # 모든 커밋을 받아야 origin/main 과 비교 가능(기본 1은 마지막 커밋만)
      - name: uv 설치
        uses: astral-sh/setup-uv@v10.2.0
        with:
          python-version: "3.12"
      - name: 라이브러리 설치
        run: uv sync --locked --group review      # 기본 묶음에 review 묶음(langchain)까지 설치
      - name: AI 리뷰 생성
        env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}   # 4-2 처럼 등록한 비밀 변수
        run: uv run python tools/ai_reviewer.py   # review.md 를 만듦
      - name: pull request 댓글로 올리기
        env:
          GH_TOKEN: ${{ github.token }}           # 실행마다 GitHub 가 주는 토큰. gh 명령이 이 값으로 로그인
        run: gh pr comment ${{ github.event.pull_request.number }} --body-file review.md
```